# Preliminary Results — First Region per Country

Gathers all `results.json` files from the first region of every country and visualises
how the once-in-100-year discharge event shifts across climate scenarios.

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from rich import print

## 1. Gather results

In [ ]:
base_path = Path('regions')

all_results = []
for country_dir in sorted(base_path.iterdir()):
    if not country_dir.is_dir():
        continue
    country = country_dir.name
    region_dirs = sorted([d for d in country_dir.iterdir() if d.is_dir()])
    if not region_dirs:
        continue
    first_region = region_dirs[0]
    results_file = first_region / 'results.json'
    if results_file.exists():
        with open(results_file) as f:
            data = json.load(f)
        data['country'] = country
        all_results.append(data)
        print(f'[green]✓[/green] {country:30s} → {first_region.name}')
    else:
        print(f'[yellow]–[/yellow] {country:30s} → {first_region.name}  [dim](no results yet)[/dim]')

print(f'\n[bold]{len(all_results)} region(s) with results loaded.[/bold]')

## 2. Parse return-period data

In [ ]:
def parse_rp(value):
    """Return (mean, std) for a return-period value.
    
    Values can be a plain number or a 'mean ± std' string.
    """
    if isinstance(value, (int, float)):
        return float(value), 0.0
    s = str(value)
    if '\u00b1' in s:          # ±
        parts = s.split('\u00b1')
        return float(parts[0].strip()), float(parts[1].strip())
    return float(s.strip()), 0.0


def get_scenario_label(key):
    """Map a results.json key to a short display label and display order."""
    k = key.lower()
    if key == 'observed_reference':
        return 'Observed\n(ref)', 0
    if 'modelled discharge' in k or ('cmip' in k and 'ssp' not in k):
        return 'CMIP6\nhist', 1
    if k == 'era5':
        return 'ERA5', 2
    if 'destine' in k and ('hist' in k or 'historical' in k):
        return 'DestinE\nhist', 3
    if 'ssp126' in k:
        return 'SSP1-2.6', 4
    if 'ssp245' in k:
        return 'SSP2-4.5', 5
    if 'ssp370' in k:
        return 'SSP3-7.0', 6
    if 'ssp585' in k:
        return 'SSP5-8.5', 7
    if 'destine' in k:
        return 'DestinE\nfuture', 8
    return key, 99


SCENARIO_COLORS = {
    'Observed\n(ref)':  '#000000',
    'CMIP6\nhist':      '#4393c3',
    'ERA5':             '#1f77b4',
    'DestinE\nhist':    '#00BCD4',
    'SSP1-2.6':         '#2ca02c',
    'SSP2-4.5':         '#bcbd22',
    'SSP3-7.0':         '#ff7f0e',
    'SSP5-8.5':         '#d62728',
    'DestinE\nfuture':  '#9467bd',
}


# Build a tidy DataFrame: one row per (region × scenario)
rows = []
for r in all_results:
    rp_data = r.get('return_periods', {})
    for key, vals in rp_data.items():
        if key == 'observed_reference':
            # The reference point is always 100 years by definition
            rows.append({
                'country':    r['country'],
                'caravan_id': r['caravan_id'],
                'scenario_key':   key,
                'scenario_label': 'Observed\n(ref)',
                'order':          0,
                'rp_mean':        100.0,
                'rp_std':         0.0,
            })
            continue
        rp_val = vals.get('rp_at_obs_q100')
        if rp_val is None:
            continue
        mean, std = parse_rp(rp_val)
        label, order = get_scenario_label(key)
        rows.append({
            'country':        r['country'],
            'caravan_id':     r['caravan_id'],
            'scenario_key':   key,
            'scenario_label': label,
            'order':          order,
            'rp_mean':        mean,
            'rp_std':         std,
        })

df = pd.DataFrame(rows).sort_values(['country', 'order']).reset_index(drop=True)
print(df[['country', 'caravan_id', 'scenario_label', 'rp_mean', 'rp_std']])

## 3. Plot — shift of the once-in-100-year discharge

In [ ]:
# Ordered scenario labels for the x-axis
ordered_labels = [
    'Observed\n(ref)',
    'CMIP6\nhist',
    'ERA5',
    'DestinE\nhist',
    'SSP1-2.6',
    'SSP2-4.5',
    'SSP3-7.0',
    'SSP5-8.5',
    'DestinE\nfuture',
]
label_to_x = {lbl: i for i, lbl in enumerate(ordered_labels)}

# One colour per country so lines are distinguishable
countries = df['country'].unique()
country_cmap = plt.cm.get_cmap('tab20', len(countries))
country_color = {c: country_cmap(i) for i, c in enumerate(countries)}

fig, ax = plt.subplots(figsize=(13, 6))

# Reference line
ax.axhline(100, color='black', linestyle='--', linewidth=1, alpha=0.4, zorder=1)

# Shaded background: left = historical, right = future
ax.axvspan(-0.5, 3.5, color='#e8f4f8', alpha=0.4, zorder=0)
ax.axvspan(3.5, len(ordered_labels) - 0.5, color='#fff3e0', alpha=0.4, zorder=0)
ax.text(1.5, ax.get_ylim()[1] if ax.get_ylim()[1] != 1.0 else 200,
        'Historical', ha='center', va='top', color='#555', fontsize=9, style='italic')

# Plot per country
for country, group in df.groupby('country'):
    color = country_color[country]
    x_vals, y_vals, y_errs = [], [], []
    for _, row in group.iterrows():
        lbl = row['scenario_label']
        if lbl in label_to_x:
            x_vals.append(label_to_x[lbl])
            y_vals.append(row['rp_mean'])
            y_errs.append(row['rp_std'])

    ax.plot(x_vals, y_vals, 'o-', color=color, linewidth=1.8,
            markersize=6, label=country.replace('_', ' ').title(), zorder=3)

    for x, y, ye in zip(x_vals, y_vals, y_errs):
        if ye > 0:
            ax.errorbar(x, y, yerr=ye, fmt='none', ecolor=color,
                        elinewidth=1.2, capsize=4, alpha=0.7, zorder=2)

ax.set_xticks(range(len(ordered_labels)))
ax.set_xticklabels(ordered_labels, fontsize=10)
ax.set_ylabel('Return period of observed Q$_{100}$ (years)', fontsize=11)
ax.set_title(
    'Shift of the once-in-100-year discharge across climate scenarios\n'
    '(first region per country)',
    fontsize=12,
)
ax.set_xlim(-0.5, len(ordered_labels) - 0.5)
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(True, which='major', alpha=0.3)
ax.grid(True, which='minor', alpha=0.15)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9,
          framealpha=0.8, title='Country')

plt.tight_layout()
plt.savefig('preliminary_q100_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preliminary_q100_shift.png')

## 4. Summary table

In [ ]:
pivot = df.pivot_table(
    index='country',
    columns='scenario_label',
    values='rp_mean',
    aggfunc='first',
)

# Reorder columns
present_cols = [c for c in ordered_labels if c in pivot.columns]
pivot = pivot[present_cols]
pivot.index = pivot.index.str.replace('_', ' ').str.title()

pivot.style \
    .format('{:.1f}') \
    .background_gradient(cmap='RdYlGn_r', axis=None, vmin=50, vmax=200) \
    .set_caption(
        'Return period (years) of the historically-observed 100-year discharge'
        ' — values > 100 indicate that the event becomes rarer; < 100 means more frequent'
    )